# 🧠 Deep Learning Training Pipeline — CIC-IDS-2018
## Phase 2: Optimization — PCA Dimensionality Reduction

| Fix | Description |
|-----|-------------|
| ✅ Correct variable names | CNN metrics use `bal_acc_cnn`, not undefined `precision_cnn` etc. |
| ✅ LSTM balanced metrics | `bal_acc_lstm`, `benign_recall_lstm` … all computed |
| ✅ No duplicate saves | Scaler/PCA saved once, not twice |
| ✅ Native Keras format | Models saved as `.keras` (no legacy HDF5 warnings) |
| ✅ No trailing separator | `os.path.join` without trailing `''` |
| ✅ CNN uses full dataset | `sample_size = None` so helper function works as designed |
| ✅ Consistent patience | `ReduceLROnPlateau` respects `PATIENCE` variable |
| ✅ Single training loop | One loop trains both CNN and LSTM (cleaner, less repetition) |

**Optimization:** PCA — StandardScaler → PCA (57 → 30 components)  
**Models:** 1D-CNN · LSTM

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, BatchNormalization,
    Dropout, Dense, Flatten, LSTM,
    Reshape
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    balanced_accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)
tf.random.set_seed(42)

# ── GPU Configuration ─────────────────────────────────────────────────────────
# Enable memory growth: prevents TF from allocating ALL GPU memory at startup.
# Without this, a single process can OOM-kill other GPU workloads.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        # Optional: restrict to a single GPU if multiple are present
        # tf.config.set_visible_devices(gpus[0], 'GPU')
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f'✅ GPU enabled: {len(gpus)} Physical GPU(s), '
              f'{len(logical_gpus)} Logical GPU(s)')
        for g in gpus:
            print(f'   {g.name}')
    except RuntimeError as e:
        # Memory growth must be set before GPUs are initialised
        print(f'⚠️  GPU config error: {e}')
else:
    print('⚠️  No GPU detected — running on CPU.')
    print('   Training will be significantly slower, especially for LSTM.')
    print('   Consider using Google Colab (free T4) or Kaggle (free P100).')

print()
print('=' * 80)
print('🧠 DEEP LEARNING TRAINING — CIC-IDS-2018 | PCA OPTIMIZATION')
print('=' * 80)
print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPU devices: {gpus}')

## 2. Configuration

In [ ]:
INPUT_FILE     = 'archive/full_df_binary_labels.csv'
BASE_MODEL_DIR = 'trained_models/dl_optimized/pca/'
TEST_SIZE      = 0.2
RANDOM_STATE   = 42
SAVE_MODELS    = True

# ── PCA configuration ────────────────────────────────────────────────────
N_COMPONENTS = 30   # Reduce 57 features → 30 principal components

# ── Training hyperparameters ─────────────────────────────────────────────
BATCH_SIZE       = 1024
EPOCHS           = 30
VALIDATION_SPLIT = 0.1
PATIENCE         = 5

# ── Per-model sample sizes ───────────────────────────────────────────────
# None  → use full dataset (let the helper function handle the logic)
MODEL_SAMPLE_SIZES = {
    'CNN' : None,    # Full dataset
    'LSTM': 200_000, # LSTM is slow — use 200 k rows
}

os.makedirs(BASE_MODEL_DIR, exist_ok=True)

print('✅ Configuration set')
print(f'   PCA components    : {N_COMPONENTS}')
print(f'   Batch size        : {BATCH_SIZE}')
print(f'   Max epochs        : {EPOCHS}')
print(f'   Validation split  : {VALIDATION_SPLIT}')
print(f'   EarlyStopping pat.: {PATIENCE}')
print(f'   Output directory  : {BASE_MODEL_DIR}')
print(f'\n📊 Per-model sample sizes:')
for model, size in MODEL_SAMPLE_SIZES.items():
    s = 'Full dataset' if size is None else f'{size:,} rows'
    print(f'   {model:<6}: {s}')

## 3. Load Dataset

In [ ]:
CHUNK_SIZE = 100_000
chunks, total = [], 0

print(f'\n📂 Loading data from: {INPUT_FILE}')
print('⏳ Reading in chunks...\n')

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    chunks.append(chunk)
    total += len(chunk)
    if (i + 1) % 20 == 0:
        print(f'   Chunk {i+1:>4}: {len(chunk):,} rows | Total: {total:,}')

df_full = pd.concat(chunks, ignore_index=True)
del chunks

df_full = df_full.drop(columns=['Label'], errors='ignore')

print(f'\n✅ Dataset loaded')
print(f'   Rows   : {len(df_full):,}')
print(f'   Columns: {len(df_full.columns)}')
print(f'\nRaw class distribution:')
print(df_full['Label_Binary'].value_counts())

## 4. Prepare Features and Target

In [ ]:
label_columns   = ['Label', 'Label_Binary']
existing_labels = [c for c in label_columns if c in df_full.columns]

X_full = df_full.drop(columns=existing_labels).select_dtypes(include=[np.number])
y_full = df_full['Label_Binary']
del df_full

print(f'✅ Features (X): {X_full.shape}')
print(f'✅ Target   (y): {y_full.shape}')
print(f'\n📊 Original features : {X_full.shape[1]}')
print(f'📊 After PCA         : {N_COMPONENTS} components '
      f'(reduction: {X_full.shape[1] - N_COMPONENTS} features)')

print(f'\n📊 Class Distribution:')
vc = y_full.value_counts()
for k, v in vc.items():
    label = 'Benign' if k == 0 else 'Attack'
    print(f'   {label} ({k}): {v:>10,}  ({v/len(y_full)*100:.2f}%)')

## 5. Helper Functions

In [ ]:
def get_model_data(X_full, y_full, model_name, sample_size,
                   test_size, random_state):
    """
    Stratified sample (optional) → train/test split.
    Returns raw numpy arrays — scaling + PCA applied later per model.
    """
    if sample_size is None or sample_size >= len(X_full):
        if sample_size is not None:
            print(f'   ⚠️  Requested {sample_size:,} rows but only '
                  f'{len(X_full):,} available — using full dataset')
        X, y = X_full, y_full
        print(f'   📊 Using full dataset: {len(X):,} rows')
    else:
        print(f'   📊 Sampling {sample_size:,} rows from {len(X_full):,} '
              f'(stratified)')
        X, _, y, _ = train_test_split(
            X_full, y_full,
            train_size=sample_size,
            random_state=random_state,
            stratify=y_full
        )

    print(f'\n   📊 Class distribution in sample:')
    for k, v in y.value_counts().items():
        label = 'Benign' if k == 0 else 'Attack'
        print(f'      {label} ({k}): {v:>10,}  ({v/len(y)*100:.2f}%)')

    print(f'\n   ✂️  Splitting train/test ({test_size*100:.0f}% test)...')
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    print(f'   ✅ Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
    return X_train, X_test, y_train, y_test


def apply_pca_pipeline(X_train, X_test, n_components, random_state):
    """
    StandardScaler → PCA.
    Fit on train only — no data leakage.
    Returns transformed arrays + fitted transformers.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    pca = PCA(n_components=n_components, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca  = pca.transform(X_test_scaled)

    print(f'   Original dims  : {X_train.shape[1]}')
    print(f'   Reduced dims   : {X_train_pca.shape[1]}')
    print(f'   Variance kept  : {pca.explained_variance_ratio_.sum()*100:.2f}%')
    print(f'   Features removed: {X_train.shape[1] - X_train_pca.shape[1]}')
    return X_train_pca, X_test_pca, scaler, pca


print('✅ Helper functions defined')

## 6. Model Architectures

### CNN — 1D Convolutional Neural Network
```
Input (30,) → Reshape (30, 1)
→ Conv1D(64,  k=3, relu, same) → BN → MaxPool(2) → Dropout(0.3)
→ Conv1D(128, k=3, relu, same) → BN → MaxPool(2) → Dropout(0.3)
→ Flatten → Dense(128, relu) → BN → Dropout(0.4)
→ Dense(64,  relu) → Dropout(0.3) → Dense(1, sigmoid)
```

### LSTM — Long Short-Term Memory
```
Input (30,) → Reshape (30, 1)
→ LSTM(128, return_sequences=True) → Dropout(0.3) → BN
→ LSTM(64,  return_sequences=False) → Dropout(0.3) → BN
→ Dense(64,  relu) → Dropout(0.4)
→ Dense(32,  relu) → Dropout(0.3) → Dense(1, sigmoid)
```

In [ ]:
def build_cnn_model(n_components):
    """1D CNN for PCA-reduced network traffic classification."""
    model = Sequential([
        Reshape((n_components, 1), input_shape=(n_components,)),

        # Conv Block 1
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Conv Block 2
        Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Dense head
        Flatten(),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model


def build_lstm_model(n_components):
    """LSTM for PCA-reduced network traffic classification."""
    model = Sequential([
        Reshape((n_components, 1), input_shape=(n_components,)),

        # LSTM layers
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        BatchNormalization(),

        LSTM(64, return_sequences=False),
        Dropout(0.3),
        BatchNormalization(),

        # Dense head
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model


MODEL_BUILDERS = {
    'CNN' : build_cnn_model,
    'LSTM': build_lstm_model,
}

print(f'✅ Defined {len(MODEL_BUILDERS)} model builders: {list(MODEL_BUILDERS.keys())}')

## 7. Train All Models

In [ ]:
results    = {}
histories  = {}
pca_objects = {}   # store pca per model for the analysis cell

for model_name, model_builder in MODEL_BUILDERS.items():
    print('\n' + '=' * 80)
    print(f'🚀 TRAINING: {model_name} WITH PCA')
    print('=' * 80)

    # No trailing '' — os.path.join handles the separator
    model_dir = os.path.join(BASE_MODEL_DIR, model_name.lower())
    os.makedirs(model_dir, exist_ok=True)
    print(f'📁 Model directory: {model_dir}')

    sample_size = MODEL_SAMPLE_SIZES.get(model_name)

    # ── Prepare raw data ──────────────────────────────────────────────────
    print(f'\n📂 Preparing data for {model_name}...')
    X_train_raw, X_test_raw, y_train, y_test = get_model_data(
        X_full, y_full, model_name, sample_size, TEST_SIZE, RANDOM_STATE
    )

    # ── StandardScaler + PCA (fit on train only) ──────────────────────────
    print(f'\n🔬 Applying scaling + PCA (fit on train only — no leakage)...')
    X_train_pca, X_test_pca, scaler, pca = apply_pca_pipeline(
        X_train_raw, X_test_raw, N_COMPONENTS, RANDOM_STATE
    )
    pca_objects[model_name] = pca

    # ── Save transformers (once — no duplicate) ───────────────────────────
    if SAVE_MODELS:
        with open(os.path.join(model_dir, 'scaler.pkl'), 'wb') as f:
            pickle.dump(scaler, f)
        with open(os.path.join(model_dir, 'pca.pkl'), 'wb') as f:
            pickle.dump(pca, f)
        print(f'   💾 Scaler & PCA saved')

    # ── Build model ───────────────────────────────────────────────────────
    print(f'\n🔨 Building {model_name} model (input: {N_COMPONENTS} components)...')
    model = model_builder(N_COMPONENTS)
    model.summary()

    # ── Callbacks ─────────────────────────────────────────────────────────
    best_model_path = os.path.join(model_dir, 'best_model.keras')
    cb_list = [
        EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=max(2, PATIENCE // 3),   # consistent with PATIENCE variable
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=best_model_path,
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
    ]

    # ── Train ─────────────────────────────────────────────────────────────
    print(f'\n⏱️  Training {model_name} with PCA-reduced features...')
    start_time = time.time()

    history = model.fit(
        X_train_pca, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        callbacks=cb_list,
        verbose=1
    )

    train_time = time.time() - start_time
    print(f'\n✅ Training completed in {train_time:.2f}s ({train_time/60:.2f} min)')

    # ── Evaluate ──────────────────────────────────────────────────────────
    print(f'\n🔮 Evaluating on test set...')
    start_eval = time.time()

    y_pred_prob = model.predict(X_test_pca, batch_size=BATCH_SIZE, verbose=0)
    y_pred      = (y_pred_prob > 0.5).astype(int).flatten()
    pred_time   = time.time() - start_eval
    print(f'✅ Predictions completed in {pred_time:.2f}s')

    # ── Compute all metrics ───────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = cm[0][0], cm[0][1], cm[1][0], cm[1][1]

    accuracy         = accuracy_score(y_test, y_pred)
    f1_macro         = f1_score(y_test, y_pred, average='macro',  zero_division=0)
    bal_acc          = balanced_accuracy_score(y_test, y_pred)
    benign_recall    = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    attack_recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    benign_precision = TN / (TN + FN) if (TN + FN) > 0 else 0.0
    attack_precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0

    print(f'\n📊 RESULTS WITH PCA:')
    print(f'   Balanced Accuracy  : {bal_acc:.4f}  ← primary metric')
    print(f'   F1 Macro           : {f1_macro:.4f}')
    print(f'   Benign Recall      : {benign_recall:.4f}')
    print(f'   Attack Recall      : {attack_recall:.4f}')
    print(f'   Benign Precision   : {benign_precision:.4f}')
    print(f'   Attack Precision   : {attack_precision:.4f}')
    print(f'   Plain Accuracy     : {accuracy:.4f}  ⚠️  (misleading on imbalanced data)')
    print(f'   Variance Explained : {pca.explained_variance_ratio_.sum()*100:.2f}%')

    print(f'\n🕐 TIME:')
    print(f'   Train: {train_time:.2f}s ({train_time/60:.2f} min)')
    print(f'   Eval : {pred_time:.2f}s')

    print(f'\n🎯 Confusion Matrix:')
    print(cm)
    print(f'   TN={TN:,}  FP={FP:,}  FN={FN:,}  TP={TP:,}')

    print(f'\n📋 Classification Report:')
    print(classification_report(y_test, y_pred,
                                target_names=['Benign', 'Attack'], zero_division=0))

    # ── Store results ─────────────────────────────────────────────────────
    results[model_name] = {
        'balanced_accuracy': float(bal_acc),
        'f1_macro'         : float(f1_macro),
        'benign_recall'    : float(benign_recall),
        'attack_recall'    : float(attack_recall),
        'benign_precision' : float(benign_precision),
        'attack_precision' : float(attack_precision),
        'accuracy'         : float(accuracy),
        'train_time'       : train_time,
        'pred_time'        : pred_time,
        'confusion_matrix' : cm,
        'sample_size'      : len(X_train_raw) + len(X_test_raw),
        'train_samples'    : len(X_train_raw),
        'test_samples'     : len(X_test_raw),
        'epochs_trained'   : len(history.history['loss']),
        'variance_explained': float(pca.explained_variance_ratio_.sum()),
        'TN': int(TN), 'FP': int(FP), 'FN': int(FN), 'TP': int(TP)
    }
    histories[model_name] = history

    # ── Save model + metadata ─────────────────────────────────────────────
    if SAVE_MODELS:
        final_path = os.path.join(model_dir, 'final_model.keras')   # native Keras format
        model.save(final_path)
        print(f'\n💾 Final model saved : {final_path}')
        print(f'💾 Best model saved  : {best_model_path}')

        with open(os.path.join(model_dir, 'history.pkl'), 'wb') as f:
            pickle.dump(history.history, f)

        metadata = {
            'model_name'      : model_name,
            'optimization'    : 'PCA',
            'n_components'    : N_COMPONENTS,
            'sample_size'     : sample_size,
            'train_samples'   : len(X_train_raw),
            'test_samples'    : len(X_test_raw),
            'epochs'          : EPOCHS,
            'epochs_trained'  : len(history.history['loss']),
            'batch_size'      : BATCH_SIZE,
            'train_time'      : train_time,
            'metrics': {
                'balanced_accuracy': float(bal_acc),
                'f1_macro'         : float(f1_macro),
                'benign_recall'    : float(benign_recall),
                'attack_recall'    : float(attack_recall),
                'accuracy'         : float(accuracy),
                'variance_explained': float(pca.explained_variance_ratio_.sum()),
            }
        }
        with open(os.path.join(model_dir, 'metadata.pkl'), 'wb') as f:
            pickle.dump(metadata, f)
        print(f'💾 Metadata saved    : {os.path.join(model_dir, "metadata.pkl")}')

print('\n' + '=' * 80)
print('✅ ALL DEEP LEARNING MODELS TRAINED & TESTED WITH PCA!')
print('=' * 80)

## 8. Results Summary

In [ ]:
summary_df = pd.DataFrame(results).T
metric_cols = [
    'balanced_accuracy', 'f1_macro',
    'benign_recall', 'attack_recall',
    'benign_precision', 'attack_precision',
    'accuracy', 'variance_explained', 'train_time', 'epochs_trained', 'sample_size'
]
summary_df = summary_df[metric_cols].astype({c: float for c in metric_cols})
summary_df = summary_df.sort_values('balanced_accuracy', ascending=False)

print('\n' + '=' * 80)
print('📊 FINAL RESULTS — PCA OPTIMIZATION')
print('=' * 80)
print('\nRanked by Balanced Accuracy:')
print(summary_df[[
    'balanced_accuracy', 'f1_macro',
    'benign_recall', 'attack_recall',
    'benign_precision', 'attack_precision',
    'accuracy'
]].to_string())

print('\n⚠️  Balanced Accuracy = (Benign Recall + Attack Recall) / 2')
print('    Benign Recall → legitimate flows correctly identified')
print('    Attack Recall → attacks correctly detected')
print('    Plain accuracy is misleading on imbalanced data.')

summary_df.to_csv(os.path.join(BASE_MODEL_DIR, 'results_summary.csv'))
print(f'\n✅ Results saved → {BASE_MODEL_DIR}results_summary.csv')
summary_df

## 9. Training History — Loss & Accuracy Curves

In [ ]:
print('\n📊 Plotting training history...')

n_models = len(histories)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 6 * n_models))
if n_models == 1:
    axes = axes.reshape(1, -1)

for idx, (model_name, history) in enumerate(histories.items()):
    h = history.history
    epochs_range = range(1, len(h['loss']) + 1)

    # Loss
    axes[idx, 0].plot(epochs_range, h['loss'],     label='Train Loss',
                      color='#2196F3', linewidth=2)
    axes[idx, 0].plot(epochs_range, h['val_loss'], label='Val Loss',
                      color='#F44336', linewidth=2, linestyle='--')
    best_ep = np.argmin(h['val_loss']) + 1
    axes[idx, 0].axvline(best_ep, color='green', linestyle=':', alpha=0.7,
                          label=f'Best epoch={best_ep}')
    axes[idx, 0].set_title(f'{model_name} — Loss', fontweight='bold', fontsize=12)
    axes[idx, 0].set_xlabel('Epoch')
    axes[idx, 0].set_ylabel('Loss')
    axes[idx, 0].legend(fontsize=9)
    axes[idx, 0].grid(alpha=0.3)

    # Accuracy
    axes[idx, 1].plot(epochs_range, h['accuracy'],     label='Train Acc',
                      color='#2196F3', linewidth=2)
    axes[idx, 1].plot(epochs_range, h['val_accuracy'], label='Val Acc',
                      color='#F44336', linewidth=2, linestyle='--')
    axes[idx, 1].set_title(f'{model_name} — Accuracy', fontweight='bold', fontsize=12)
    axes[idx, 1].set_xlabel('Epoch')
    axes[idx, 1].set_ylabel('Accuracy')
    axes[idx, 1].legend(fontsize=9)
    axes[idx, 1].grid(alpha=0.3)
    axes[idx, 1].set_ylim(0.9, 1.0)

plt.suptitle(
    'Training History — DL Models with PCA Optimization',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
hist_path = os.path.join(BASE_MODEL_DIR, 'training_history.png')
plt.savefig(hist_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {hist_path}')
plt.show()

## 10. Performance Comparison

In [ ]:
print('\n📊 Creating performance comparison plots...')

model_names = summary_df.index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'DL Models Performance — PCA Optimization (57 → 30 components)',
    fontsize=13, fontweight='bold'
)

# ── Balanced Accuracy ──
colors_ba = ['#2ecc71' if v >= 0.95 else '#f39c12' if v >= 0.90 else '#e74c3c'
             for v in summary_df['balanced_accuracy']]
axes[0, 0].barh(model_names, summary_df['balanced_accuracy'], color=colors_ba)
axes[0, 0].set_xlabel('Balanced Accuracy')
axes[0, 0].set_title('Balanced Accuracy (Primary Metric)', fontweight='bold')
axes[0, 0].set_xlim(0, 1)
axes[0, 0].axvline(0.95, color='green', linestyle='--', alpha=0.6, label='0.95 target')
axes[0, 0].axvline(0.5,  color='red',   linestyle='--', alpha=0.4, label='Random baseline')
for i, v in enumerate(summary_df['balanced_accuracy']):
    axes[0, 0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10)
axes[0, 0].legend(fontsize=9)

# ── Per-class Recall ──
x     = np.arange(len(model_names))
width = 0.35
axes[0, 1].bar(x - width/2, summary_df['benign_recall'], width,
               label='Benign Recall', color='steelblue')
axes[0, 1].bar(x + width/2, summary_df['attack_recall'], width,
               label='Attack Recall', color='coral')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].set_title('Per-Class Recall (Critical for IDS)', fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(model_names, fontsize=10)
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].axhline(0.95, color='green', linestyle='--', alpha=0.5, label='0.95 target')
axes[0, 1].legend(fontsize=9)

# ── F1 Macro ──
axes[1, 0].barh(model_names, summary_df['f1_macro'], color='mediumpurple')
axes[1, 0].set_xlabel('F1 Macro')
axes[1, 0].set_title('F1 Macro Score', fontweight='bold')
axes[1, 0].set_xlim(0, 1)
for i, v in enumerate(summary_df['f1_macro']):
    axes[1, 0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10)

# ── Training Time ──
axes[1, 1].barh(model_names, summary_df['train_time'] / 60, color='lightgreen')
axes[1, 1].set_xlabel('Training Time (minutes)')
axes[1, 1].set_title('Training Time', fontweight='bold')
for i, v in enumerate(summary_df['train_time'] / 60):
    axes[1, 1].text(v + 0.05, i, f'{v:.1f} min', va='center', fontsize=10)

plt.tight_layout()
comp_path = os.path.join(BASE_MODEL_DIR, 'metrics_comparison.png')
plt.savefig(comp_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {comp_path}')
plt.show()

## 11. Confusion Matrices

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for idx, model_name in enumerate(summary_df.index):
    r   = results[model_name]
    cm  = r['confusion_matrix']
    bal = r['balanced_accuracy']
    br  = r['benign_recall']
    ar  = r['attack_recall']

    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
        xticklabels=['Benign', 'Attack'],
        yticklabels=['Benign', 'Attack']
    )
    axes[idx].set_title(
        f'{model_name}\nBal.Acc={bal:.4f}\n'
        f'B.Rec={br:.4f} | A.Rec={ar:.4f}',
        fontweight='bold', fontsize=9
    )
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle(
    'Confusion Matrices — DL Models with PCA',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
cm_path = os.path.join(BASE_MODEL_DIR, 'confusion_matrices.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {cm_path}')
plt.show()

## 12. PCA Analysis

In [ ]:
# Use CNN's PCA for analysis (both models trained on same N_COMPONENTS)
pca_to_analyze = pca_objects.get('CNN', list(pca_objects.values())[0])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scree plot
axes[0].plot(range(1, len(pca_to_analyze.explained_variance_ratio_) + 1),
             pca_to_analyze.explained_variance_ratio_, 'bo-', linewidth=2)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained Ratio')
axes[0].set_title('Scree Plot — Variance per Component', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Cumulative variance
cumsum = np.cumsum(pca_to_analyze.explained_variance_ratio_)
axes[1].plot(range(1, len(cumsum) + 1), cumsum, 'ro-', linewidth=2)
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
axes[1].axhline(y=0.99, color='b', linestyle='--', alpha=0.6, label='99% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Cumulative Variance Explained', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('PCA Analysis — Feature Variance Decomposition',
             fontsize=13, fontweight='bold')
plt.tight_layout()
pca_path = os.path.join(BASE_MODEL_DIR, 'pca_analysis.png')
plt.savefig(pca_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {pca_path}')
plt.show()

print(f'\n📊 PCA Analysis:')
print(f'   Components selected  : {N_COMPONENTS}')
print(f'   Total variance kept  : {cumsum[N_COMPONENTS-1]*100:.2f}%')
print(f'   Top  5 components    : {cumsum[4]*100:.2f}%')
print(f'   Top 10 components    : {cumsum[9]*100:.2f}%')
print(f'   Top 20 components    : {cumsum[19]*100:.2f}%')

## 13. Final Summary

In [ ]:
print('\n' + '=' * 80)
print('🎉 TRAINING COMPLETE — PCA OPTIMIZATION')
print('=' * 80)

print('\n📋 Fixes applied in this notebook:')
print('   ✅ Correct variable names    (CNN metrics used bal_acc_cnn, not undefined vars)')
print('   ✅ LSTM balanced metrics     (bal_acc, benign/attack recall all computed)')
print('   ✅ No duplicate saves        (scaler/pca saved once per model)')
print('   ✅ Native Keras format       (.keras — no legacy HDF5 warnings)')
print('   ✅ No trailing separator     (os.path.join without trailing empty string)')
print('   ✅ CNN uses full dataset     (sample_size=None)')
print('   ✅ Consistent patience       (ReduceLROnPlateau derived from PATIENCE)')
print('   ✅ Single training loop      (no repeated code for CNN vs LSTM)')

print(f'\n📊 Models Ranked by Balanced Accuracy:')
print(f'   {"Rank":<4} {"Model":<6} {"Bal.Acc":>8} '
      f'{"B.Recall":>9} {"A.Recall":>9} {"F1-Mac":>7} '
      f'{"VarExp":>7} {"Epochs":>7} {"Time":>8}')
print('   ' + '─' * 72)
for rank, (model_name, row) in enumerate(summary_df.iterrows(), 1):
    t_str = f'{row["train_time"]/60:.1f}min'
    print(f'   {rank:<4} {model_name:<6} '
          f'{row["balanced_accuracy"]:>8.4f} '
          f'{row["benign_recall"]:>9.4f} '
          f'{row["attack_recall"]:>9.4f} '
          f'{row["f1_macro"]:>7.4f} '
          f'{row["variance_explained"]*100:>6.2f}% '
          f'{int(row["epochs_trained"]):>7} '
          f'{t_str:>8}')

best = summary_df.index[0]
print(f'\n🏆 Best model: {best}')
print(f'   Balanced Accuracy : {summary_df.loc[best, "balanced_accuracy"]:.4f}')
print(f'   Benign Recall     : {summary_df.loc[best, "benign_recall"]:.4f}')
print(f'   Attack Recall     : {summary_df.loc[best, "attack_recall"]:.4f}')
print(f'   Variance Explained: {summary_df.loc[best, "variance_explained"]*100:.2f}%')

print(f'\n📁 All models saved to: {BASE_MODEL_DIR}')
for model_name in MODEL_BUILDERS:
    d = os.path.join(BASE_MODEL_DIR, model_name.lower())
    print(f'   {d}/')
    print(f'     best_model.keras   ← best val_loss checkpoint')
    print(f'     final_model.keras  ← end-of-training weights')
    print(f'     scaler.pkl')
    print(f'     pca.pkl')
    print(f'     history.pkl')
    print(f'     metadata.pkl')

print('\n💡 Key reminder:')
print('   A good IDS needs HIGH recall on BOTH classes.')
print('   Low Benign Recall → too many false alarms (legitimate traffic blocked).')
print('   Low Attack Recall → attacks slipping through undetected.')
print(f'\n   PCA: 57 original features → {N_COMPONENTS} components used.')